# Quickstart — Datos México Python SDK

**Datos México** es un observatorio independiente de datos públicos mexicanos, mantenido por estudiantes y egresados del ITAM. Este SDK expone los datasets del observatorio (CDMX servidores públicos, CONSAR/SAR, ENIGH 2024 NS y endpoints comparativos cross-dataset) como una API Python tipada con Pydantic.

**Qué cubre el SDK** (78 endpoints en total):

- `cdmx`: padrón de servidores públicos del Gobierno de la CDMX.
- `consar`: serie histórica del Sistema de Ahorro para el Retiro.
- `enigh`: hogares mexicanos según la ENIGH 2024 Nueva Serie.
- `comparativo`: cruces editoriales CDMX × CONSAR × ENIGH.
- `personas` / `nombramientos` / `demo` / `export`: utilidades.

**Pre-requisitos**: Python 3.10+ y `pip`.

Este notebook se ejecuta end-to-end en ~10 segundos contra `https://api.datos-itam.org`.

## 1. Instalación

Desde PyPI (cuando sea publicado):

```bash
pip install datos-mexico
```

O desde el repo local en modo editable:

```bash
pip install -e /ruta/al/repo
```

In [1]:
from datos_mexico import DatosMexico

client = DatosMexico()
print("Cliente listo. Conectado a https://api.datos-itam.org")

Cliente listo. Conectado a https://api.datos-itam.org


## 2. Verificar conectividad

El método `health()` consulta `GET /health` y no se cachea.

In [2]:
h = client.health()
print(f"API status: {h.status}")

API status: ok


## 3. Tres datasets en una sola sesión

El cliente comparte un único `HttpClient` con cache TTL y retries exponenciales para todos los namespaces.

In [3]:
cdmx = client.cdmx.dashboard_stats()
consar = client.consar.recursos_totales()
enigh = client.enigh.hogares_summary()

print(f"CDMX: {cdmx.total_servidores:,} servidores en el padrón")
print(f"CONSAR: {consar.n_puntos} puntos en la serie histórica del SAR")
print(f"ENIGH: {enigh.n_hogares_expandido:,} hogares en el universo ENIGH 2024 NS")

CDMX: 246,831 servidores en el padrón
CONSAR: 326 puntos en la serie histórica del SAR
ENIGH: 38,830,230 hogares en el universo ENIGH 2024 NS


## 4. Manejo de errores

Todas las excepciones del SDK heredan de `DatosMexicoError`. Un ID que no existe lanza `NotFoundError`.

In [4]:
from datos_mexico.exceptions import NotFoundError

try:
    client.cdmx.sector_stats(99_999_999)
except NotFoundError as exc:
    print(f"NotFoundError esperado: {exc}")

GET /api/v1/sectores/99999999/stats failed: HTTP 404


NotFoundError esperado: GET /api/v1/sectores/99999999/stats → HTTP 404


## 5. Cierre y siguiente paso

El cliente expone `close()` y soporta `with`. Es buena práctica cerrarlo cuando terminamos para liberar conexiones HTTP.

**Siguiente paso**: leer `02_cdmx_servidores_publicos.ipynb` para un análisis completo del padrón de servidores públicos CDMX.

In [5]:
client.close()